### Staging Layer

-- Read the file for the now as we don't have the cloud access

-- here we want to split the claim_data into the multiple dim tables

-- fact table

    claims_data

-- dim tables

    patient --> patient_id
    diagnosis_code --> Diagnosis_Code
    provider --> Provider_ID
    Procedure_Code --> Procedure_Code

-- Need to add the below mentioned columns:
- is_active --> currently available
- is_incremental  --> will mark it as a 1 if same received in the alternate load 
- data_version when file loaded in the location that day in YYYYMMDD
- update_date --> current_date
- delete_date --> when record removed 
- deletion_flag --> if recorded deleted earlier

In [0]:
%run ./HA_control

### HA_control
Reads the weekly load sheet (`Weekly_load_stats.csv`), checks it, and tells a layer notebook which tables to run.

Load it with `%run ./HA_control` as the only line of its own cell. It defines functions only, so it is safe to `%run`.


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark import pipelines as dp
import logging
import pandas as pd
import json
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    force =True
)

logger = logging.getLogger(__name__)

In [0]:
def create_df (f_type = "csv",he_p = "true",path = None):
    df = (
    spark.read
    .format(f_type)
    .option("header", he_p)
    .load(path)
    )
    new_cols = ["_".join(str(c).lower().split()) for c in df.columns]
    logger.info("data frame for the table is created")
    return df.toDF(*new_cols)


### Creating the dim tables from the fact tables

- patient --> patient_id

- diagnosis_code --> Diagnosis_Code

- provider --> Provider_ID

- Procedure_Code --> Procedure_Code

In [0]:
def norm(name):
    return "".join(c for c in str(name).lower() if c.isalnum())

In [0]:
import csv
def load_mapping(comb_path):
    map = {}
    lookup ={}
    with open(comb_path,newline="") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            key = row[0].strip()
            values = [i.strip() for i in row[1:] if i.strip()]
            map[key] = values
    for key,values in map.items():
        for v in [key]+values:
            n=norm(v)
            if n in lookup and lookup[n] !=key:
                logger.warning(f"Alias {v!r} is ambiguous: {lookup[n]} vs {key}")
            lookup[n] = key
    return lookup

In [0]:
def column_validation(col_list,comb_path):
    if not col_list:
        logger.error("No columns are provided")
        raise ValueError("No columns are provided")
    try:
        iter(col_list)
    except TypeError as e:
        logger.error(f"column list must be the iterable {e}")
        raise ValueError("col_list must be a list of column names") from e
    for i in col_list:
        if not isinstance(i,str):
            logger.error(f"Column name must be a String, got {type(i).__name__}:{i!r}")
            raise ValueError(f"Column name must be a String, got {type(i).__name__}:{i!r}")
    matched ={}
    lookup = load_mapping(comb_path)
    for col in col_list:
        expected = lookup.get(norm(col))
        if expected:
            matched[col] = expected
        else:
            logger.warning(f"No match found for columns: {col!r}")
    return matched

In [0]:
def dim_table(df,col_list,path):
    matched = column_validation(col_list,path)
    df_columns= ["_".join(str(i).lower().split(" ")) for i in df.columns]
    # print(df_columns)
    if not matched:
        logger.error("None of the provided columns matched the expected columns")
        raise ValueError("None of the provided columns matched the expected columns")
    missing = [i for i in matched.values() if i.lower() not in df_columns ]
    if missing:
        logger.error(f"Column {missing} is not in the dataframe")
        raise ValueError(f"Column {missing} is not in the dataframe")
    expected_names = list(matched.values())
    dupes = {n for n in expected_names if expected_names.count(n)>1}
    if dupes:
        raise ValueError(f"Duplicate columns found: {dupes}")
    dim_df = df.select(*[col(f"{exp.lower()}").alias(exp.lower()) for src,exp in matched.items()])
    logger.info(f"dim_df created with columns {expected_names}")
    return dim_df

In [0]:
def get_data_version(cfg):
    "data_version is set per layer in dev_config.json; fail loudly instead of guessing a value"
    v = str(cfg.get("data_version") or "").strip()
    if not v:
        raise ValueError("dev_config.json has no data_version for this layer")
    return v

def load_config(path):
    """Load JSON config, auto-repairing missing commas between key-value pairs.
    The config file in the shared volume is occasionally re-saved without commas;
    this catches the breakage, repairs it in place, and re-validates."""
    with open(path, 'r') as f:
        raw = f.read()
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        logger.warning(f"{path}: JSON parse error ({e}), attempting auto-repair")
        lines = raw.split('\n')
        fixed_lines = []
        for i, line in enumerate(lines):
            stripped = line.rstrip()
            if (stripped.endswith('"') and not stripped.endswith(',')
                    and not stripped.endswith('{') and not stripped.endswith('[')
                    and not stripped.endswith(':')
                    and i + 1 < len(lines)
                    and lines[i + 1].lstrip().startswith('"')):
                line = stripped + ','
            fixed_lines.append(line)
        fixed = '\n'.join(fixed_lines)
        with open(path, 'w') as f:
            f.write(fixed)
        logger.info(f"{path}: auto-repaired missing commas, re-validating")
        with open(path, 'r') as f:
            return json.load(f)


def config_column(df, data_version):
    new_df = (df.withColumn("updated_at",current_timestamp())
              .withColumn("data_version",lit(data_version)))
    return new_df

In [0]:
# def mapping_based_df_read_write(map_path,col_val_path,data_path):
#   map_df = spark.read.option("header","true").csv(map_path)
#   for i in map_df:

def extract_dim_from_fact(file_path,layer, enabled=None):
    
    config = load_config(file_path)
    env = config["env"]
    catalog = config[layer]["catalog"]
    schema = config[layer]["schema"]
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
    location = config[layer]["location"]
    map_path = config[layer]["mappath"]
    path_col = config[layer]["pathcol"]
    data_version = get_data_version(config[layer])
    file_path = config[layer]["filepath"]
    column_df = spark.read.option("header", "true").csv(file_path)
    map_df = spark.read.option("header", "true").csv(map_path)
    df_claims = create_df("csv", "true", file_path)
    for row in map_df.select("tgt_table","derived _table_columns", "file_extension").collect():
        tgt_table = row["tgt_table"]
        # print("tgt_table:",tgt_table)
        if enabled is not None and tgt_table.strip().lower() not in enabled:
            continue
        src_column = row["derived _table_columns"]
        # print("src_column:",src_column)
        file_extension = row["file_extension"]
        # print("src_column:",file_extension)
        if tgt_table!="mx_claim" and src_column is not None:
            dimension_df = dim_table(df_claims,[src_column],path_col).dropDuplicates()
            config_column_df = config_column(dimension_df, data_version)
            config_column_df.write.format("parquet").mode("overwrite").save(f"{location}/data/{layer}/{tgt_table}/{file_extension}")
            config_column_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.{layer}_{tgt_table}")
        elif tgt_table=="mx_claim":
            df_claims.write.format("parquet").mode("overwrite").save(f"{location}/data/{layer}/{tgt_table}/{file_extension}")
            df_claims.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.{layer}_{tgt_table}")


    

In [0]:
config_path = "/Volumes/dab/healthcare_datasets/healthcare_vol/config/dev_config.json"
layer = "staging"
extract_dim_from_fact(config_path,layer)

2026-10-26 15:11:25,878 | INFO | __main__ | data frame for the table is created
2026-10-26 15:11:32,074 | WARNING | __main__ | Alias 'pid' is ambiguous: Patient_id vs provider_id
2026-10-26 15:11:32,179 | INFO | __main__ | dim_df created with columns ['diagnosis_code']
2026-10-26 15:11:37,034 | WARNING | __main__ | Alias 'pid' is ambiguous: Patient_id vs provider_id
2026-10-26 15:11:37,035 | INFO | __main__ | dim_df created with columns ['Patient_id']
2026-10-26 15:11:41,146 | WARNING | __main__ | Alias 'pid' is ambiguous: Patient_id vs provider_id
2026-10-26 15:11:41,147 | INFO | __main__ | dim_df created with columns ['provider_id']
2026-10-26 15:11:45,731 | WARNING | __main__ | Alias 'pid' is ambiguous: Patient_id vs provider_id
2026-10-26 15:11:45,732 | INFO | __main__ | dim_df created with columns ['procedure_code']
